# YouTube → Auto-AVSR — Colab çalıştırma

Bu notebook, projeyi **GPU'lu Colab** üzerinde localdeki gibi çalıştırır: video indirir → resmî
Auto-AVSR preprocessing (RetinaFace) ile 96×96 ağız ROI üretir → transkript çıkarır →
`accepted/review/rejected` ayırır → sonucu **Hugging Face** ortak dataset'ine yükler.

**Önce:** `Runtime → Change runtime type → GPU` (T4 yeterli) seçili olmalı.

Hücreleri sırayla çalıştır. Link eklemeyi **6. adımda** yaparsın.


## 0. GPU kontrolü


In [ ]:
!nvidia-smi

## 1. Repoyu klonla

> Repo private ise klonlama için token gerekir; o durumda `git clone` satırını
> `https://<TOKEN>@github.com/...` biçimiyle çalıştır.


In [ ]:
%cd /content
!rm -rf youtube-to-autoavsr
!git clone https://github.com/iboRotti52/youtube-to-autoavsr.git
%cd youtube-to-autoavsr
!git log --oneline -3

## 2. Sistem araçları (ffmpeg + git-lfs)

RetinaFace ağırlıkları git-lfs ile indiği için git-lfs şart.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git-lfs
!git lfs install

## 3. Python paketini kur


In [ ]:
!pip install -q -e .

## 4. Auto-AVSR + RetinaFace + Whisper kurulumu

`setup-external` resmî `mpc001/auto_avsr` reposunu klonlar; `setup-retinaface`
torch + ibug yüz tespit/hizalama paketlerini + önceden eğitilmiş ağırlıkları indirir;
`setup-whisper` `large-v3-turbo` modelini önceden indirir. Bu hücre birkaç dakika sürer.


In [ ]:
!yt2avsr setup-external   --config configs/default.yaml
!yt2avsr setup-retinaface --config configs/default.yaml
!yt2avsr setup-whisper    --config configs/default.yaml

## 5. Hugging Face girişi

Çıktıyı ortak private dataset'e (`iboRotti/avsr-tr-dataset`) yüklemek için **Write**
yetkili token'ınla giriş yap. Token'ın klasör adın (`data/<HF-kullanıcı-adın>`) olarak kullanılır.

> Token'ı hiçbir dosyaya/koda yazma; aşağıdaki kutuya yapıştır.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Kaynak linklerini gir

İki liste var, aralarındaki **tek fark videoda dış ses (voice-over) olup olmaması**:

- `no_voiceover` → ekranda konuşan, dış sesi olmayan videolar
- `voiceover` → anlatıcı / dublaj içerebilen videolar (lip-sync kontrolü sıkı uygulanır)

Linkleri aşağıdaki listelere ekle (tırnak içinde, virgülle).


In [ ]:
from pathlib import Path

no_voiceover = [
    # "https://www.youtube.com/watch?v=VIDEO_ID",
]
voiceover = [
    # "https://www.youtube.com/watch?v=VIDEO_ID",
]

Path("sources_no_voiceover.txt").write_text("\n".join(no_voiceover) + "\n")
Path("sources_voiceover.txt").write_text("\n".join(voiceover) + "\n")
print("no_voiceover:", len(no_voiceover), "| voiceover:", len(voiceover))

## 7a. (İsteğe bağlı) Tek video ile hızlı test

Kurulumun çalıştığını görmek için tek bir linki işle. `VIDEO_ID`'yi değiştir.


In [ ]:
!yt2avsr process "https://www.youtube.com/watch?v=VIDEO_ID" --config configs/default.yaml

## 7b. Toplu işleme (iki source dosyası)

6. adımda link girdiğin dosyaları tek komutla işler. Boş dosya atlanır; tamamlanan
işler SQLite'ta tutulduğu için tekrar çalıştırınca baştan yapılmaz.


In [ ]:
!yt2avsr process-both-sources --config configs/default.yaml

## 8. Çıktıyı incele

Üretilen klipler `data/clips/...` altında; manifestler `data/manifests/` altında.


In [ ]:
!find data -name '*.csv' -maxdepth 4 2>/dev/null
print('--- accepted ---')
!find data -name accepted.csv -exec sh -c 'echo {}; head -n 5 {}' \; 2>/dev/null
!ls data/clips 2>/dev/null | head

## 9. Hugging Face'e yükle

Varsayılan olarak yalnızca `accepted` + `review` klipleri, sadece VSR için gerekenler
(`mouth.mp4`, `transcript.txt`, `metadata.json`) yüklenir.

- Sesli-görüntülü (AV) model için sesi de yüklemek istersen `--include-audio` ekle
- Klasör adını değiştirmek için `--contributor İSİM` ekle


In [ ]:
!yt2avsr push-data --config configs/default.yaml
# !yt2avsr push-data --config configs/default.yaml --include-audio --contributor ibrahim

## (Alternatif) Google Drive'a kalıcı kayıt

HF yerine/ek olarak veriyi Drive'da tutmak istersen aşağıyı çalıştır.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p '/content/drive/MyDrive/avsr-tr-data'
# !cp -r data '/content/drive/MyDrive/avsr-tr-data/'
# print('Drive kopyalandı')